# Lunar rover terramechanics, validated, as an animated dashboard

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PINNeAPPle-Labs/PINNeAPPle/blob/main/notebooks/colab/04_lunar_rover_dashboard.ipynb)

NASA's VIPER rover (450 kg, four 0.5 m wheels with grousers) driving on lunar regolith (GRC-1 simulant, lunar
gravity) through a crater field. Everything is a library feature:

* **soil** with each parameter's source, **Wong-Reece wheel model** (normal and shear stress on the contact arc);
* **deformable terrain** that keeps ruts and grouser marks; **multibody vehicle** with four driven wheels;
* a **PINN surrogate** of the wheel-soil contact, with a held-out error report;
* the **dashboard**: the animation in the middle with the charts synced to it, exported as a GIF.

Before the drive, the wheel model is checked against published single-wheel testbed measurements, so the dashboard
shows a model whose error is known.

In [ ]:
# Setup: on Colab (or wherever PINNeAPPle is missing) install it from PyPI (+ imageio for the GIF), then the device.
import importlib.util, os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB or importlib.util.find_spec("pinneapple") is None or importlib.util.find_spec("imageio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pinneapple>=0.7.0", "imageio"], check=True)

import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import pinneapple as pp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PINNeAPPle {pp.__version__} on {DEVICE}")

In [ ]:
# Parameters. SMOKE=True shrinks every size so the notebook runs in about a minute (the CI runs it that way).
SMOKE = os.environ.get("PINNEAPPLE_NB_SMOKE", "0") == "1"
DURATION = 3.0 if SMOKE else 30.0        # simulated drive [s]
SPEED = 0.2                              # commanded speed [m/s]
SURROGATE_EPOCHS = 100 if SMOKE else 3000   # only used when pp.zoo is not installed (0.7.0)
N_FRAMES = 8 if SMOKE else 60            # frames of the animation
FRAME_SIZE = (640, 360) if SMOKE else (960, 540)

## 1. The wheel model against single-wheel testbed data

`testbed.validate` runs the Wong-Reece model at the measured slips of a published single-wheel test, with the soil
the paper cites, and reports the errors relative to the load.

In [ ]:
from pinneapple_simulation.terramechanics import WongReeceModel, get_soil, testbed

print("bundled datasets:", testbed.list_datasets())
reports = []
for ds, wheel in testbed.list_datasets():
    rep = testbed.validate(testbed.load_dataset(ds, wheel))
    reports.append(rep)
    e = rep.errors.get("drawbar_pull_N", {})
    print(f"{rep.data.label:<24} load {rep.data.load:6.0f} N   drawbar-pull RMSE {e.get('rmse', float('nan')):6.1f} N"
          f" ({100 * e.get('rmse_rel_scale', float('nan')):.1f} % of the load)")

fig, axs = plt.subplots(1, len(reports), figsize=(4 * len(reports), 3.2), squeeze=False)
for ax, rep in zip(axs[0], reports):
    ax.plot(rep.curve["slip"] if "slip" in rep.curve else np.linspace(0, max(0.7, rep.data.slip.max()), 41),
            rep.curve["drawbar_pull"], label="Wong-Reece")
    ax.plot(rep.data.slip, rep.data.drawbar_pull, "ko", ms=4, label="measured")
    ax.set(title=rep.data.label, xlabel="slip", ylabel="drawbar pull [N]")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. The drive

`VehicleSimulator` integrates the multibody vehicle with the wheel model at every contact and stamps the terrain;
`run.summary()` gives distance, slip, sinkage, energy and loads.

In [ ]:
from pinneapple_simulation.vehicle import VehicleSimulator, build_vehicle, scenarios, viper

spec = viper()
soil = get_soil("GRC-1")
site = scenarios.crater_site()
t0 = time.time()
run = VehicleSimulator(build_vehicle(spec), site, soil).run(DURATION, speed=SPEED)
print(f"simulated {DURATION:g} s in {time.time() - t0:.0f} s")
{k: round(float(v), 4) for k, v in run.summary().items()}

### Physics checks of the run

* the rover moved forward, at a speed close to the command;
* the slip stays in the range where the wheel model was validated;
* the sinkage is positive and small compared with the wheel radius;
* the drive energy never decreases (no free energy from the soil).

In [ ]:
s = run.summary()
dist = run.distance()
E = run.energy()
slip = np.asarray(run.wheel["slip"]) if "slip" in run.wheel else None
checks = {
    "moved forward": dist[-1] > 0.3 * SPEED * DURATION,
    "speed within 50 % of the command": abs(dist[-1] / DURATION - SPEED) <= 0.5 * SPEED,
    "drive energy non-decreasing": bool(np.all(np.diff(E) >= -1e-9)),
    "sinkage below 0.2 wheel radius": float(np.nanmax(run.wheel["sinkage"])) < 0.2 * spec.wheel.radius,
}
if slip is not None:
    checks["slip within [-0.4, 0.9] (validated range)"] = bool(np.nanmax(np.abs(slip)) <= 0.9)
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")

## 3. A PINN of the wheel-soil contact, from the model zoo

`pp.zoo.load("rover_contact_grc1_viper")` is a physics-informed network of the Wong-Reece contact (drawbar pull,
normal load and torque versus slip and sinkage) for this wheel on GRC-1, verified on held-out states with its own
physics checks. On PINNeAPPle 0.7.0, which predates the zoo, the cell trains the same kind of network with
`train_contact_surrogate` instead.

In [ ]:
from pinneapple_simulation.terramechanics.surrogate import ContactSurrogate, train_contact_surrogate

wr = WongReeceModel(soil, spec.wheel)
if hasattr(pp, "zoo"):
    zm = pp.zoo.load("rover_contact_grc1_viper")
    c = zm.config
    m = zm.card.validation_metrics
    sur = ContactSurrogate(zm.module, np.array(c["x_lo"]), np.array(c["x_hi"]), np.array(c["y_lo"]), np.array(c["y_hi"]),
                           tuple(zm.bounds()["slip"]), tuple(zm.bounds()["sinkage_m"]),
                           meta={"test_errors": {"Fx": {"rmse_N": m["test/Fx_rmse_N"]}}})
    print("zoo model, held-out RMSE [N]:", {k: round(v, 3) for k, v in m.items() if k.endswith("rmse_N")})
    print("zoo physics checks:", pp.zoo.validate("rover_contact_grc1_viper").decisao)
else:
    sur, rep = train_contact_surrogate(wr, (-0.4, 0.9), (0.002, 0.06), epochs=SURROGATE_EPOCHS)
    print("held-out RMSE / range:", {k: round(v["rmse_over_range"], 4) for k, v in rep["test"].items()})

## 4. The dashboard

The frames below are drawn with matplotlib (a top view of the terrain, ruts coloured by the peak normal stress the
wheels left, and the rover's path), so the notebook needs nothing beyond PyPI. With PyVista and a display
(`pinneapple[pyvista]`), `vehicle.dashboard.render_frames(run, site)` gives the 3-D renders of the docs instead.

In [ ]:
from pinneapple_simulation.vehicle import dashboard as vdash
from pinneapple_simulation.vehicle.viz import terrain_at

ks = np.unique(np.linspace(0, len(run.t) - 1, N_FRAMES).astype(int))
xy = run.q[:, :2]
cache = {}
ter_end = terrain_at(run, site, float(run.t[-1]))
stress_max = float(np.percentile(ter_end.sigma[ter_end.sigma > 0], 99)) / 1e3 if (ter_end.sigma > 0).any() else 1.0
x0, x1 = xy[:, 0].min() - 1.5, xy[:, 0].max() + 1.5
y0, y1 = xy[:, 1].min() - 1.0, xy[:, 1].max() + 1.0

def top_view(k):
    ter = terrain_at(run, site, float(run.t[k]), cache)
    fig = plt.figure(figsize=(FRAME_SIZE[0] / 100, FRAME_SIZE[1] / 100), dpi=100)
    fig.patch.set_facecolor("black")
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_facecolor("black")
    ext = (ter.x0, ter.x1, ter.y0, ter.y1)
    lo, hi = np.percentile(ter.h0, [1, 99])
    ax.imshow(ter.h0, extent=ext, origin="lower", cmap="gray", vmin=lo, vmax=hi)   # relief: craters and rocks
    sig = np.ma.masked_less_equal(ter.sigma / 1e3, 0)
    ax.imshow(sig, extent=ext, origin="lower", cmap="inferno", vmin=0, vmax=stress_max)
    ax.plot(xy[: k + 1, 0], xy[: k + 1, 1], color="cyan", lw=1.5)
    ax.plot(*xy[k], "o", color="cyan", ms=8)
    ax.set(xlim=(x0, x1), ylim=(y0, y1))
    ax.axis("off")
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
    plt.close(fig)
    return img

frames = [top_view(int(k)) for k in ks]
plt.figure(figsize=(8, 4.5))
plt.imshow(frames[-1])
plt.axis("off")
plt.show()

In [ ]:
db = vdash.vehicle_dashboard(run, frames, ks, model=wr, surrogate=sur, clim=(0.0, stress_max),
                             title="NASA VIPER on lunar regolith",
                             subtitle="Multibody dynamics, Wong-Reece wheels on GRC-1 (lunar g), deformable regolith")
out = db.to_video(gif="rover_dashboard.gif", fps=6, gif_size=(800, 450))
db.to_png("rover_dashboard.png", k=len(ks) - 1)
print(out, os.path.getsize("rover_dashboard.gif") // 1024, "kB")

In [ ]:
from IPython.display import Image, display

display(Image(filename="rover_dashboard.gif"))

`db.to_html("rover.html")` writes the interactive version (one file, works offline): scrubbing a chart moves the
animation. In Colab, download it from the file panel on the left.